# Pseudobulk model building — WGCNA

**Environment:** `clamp-analyses`

Weighted Gene Co-expression Network Analysis on every pseudobulk dataset. Preprocesses raw counts from `bulk_expr.csv`. Soft-thresholding power chosen via `pickSoftThreshold`. Module eigengenes (B matrix, modules × samples) and kME-based loadings (Z matrix, genes × modules) are saved to `output/01_model_building/05_pseudobulk/<dataset>/WGCNA/`.

## Libraries

In [ ]:
library(data.table)
library(here)
library(CLAMP)
library(WGCNA)

enableWGCNAThreads(nThreads = 4)
set.seed(123)

## Configuration

In [ ]:
DATASET  = "PBMC_Perez2022"
OUT_ROOT = "output/01_model_building/05_pseudobulk"
DATA_DIR = "data/pseudobulk"

## Build WGCNA model for each dataset

In [ ]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)

# Load preprocessed data
norm_dt    <- fread(file.path(here(), OUT_ROOT, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
samples <- colnames(norm)
cat(DATASET, "norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

# WGCNA expects samples x genes
datExpr <- as.data.frame(t(norm))

# Soft thresholding power
powers <- 1:20
sft <- pickSoftThreshold(datExpr, powerVector = powers,
                         networkType = "unsigned", verbose = 0)
idx <- which(sft$fitIndices$SFT.R.sq >= 0.9)[1]
soft_power <- if (!is.na(idx)) sft$fitIndices$Power[idx] else 7
cat("  Soft power:", soft_power, "\n")

# Build network
message("  Running blockwiseModules ...")
net <- blockwiseModules(
  datExpr,
  power          = soft_power,
  networkType    = "unsigned",
  TOMType        = "unsigned",
  minModuleSize  = 30,
  mergeCutHeight = 0.25,
  numericLabels  = TRUE,
  verbose        = 0,
  maxBlockSize   = ncol(datExpr),
  saveTOMs       = FALSE
)

# Module eigengenes (B: modules x samples)
MEs <- net$MEs
if ("ME0" %in% colnames(MEs)) MEs <- MEs[, colnames(MEs) != "ME0"]

n_mods <- ncol(MEs)
cat("  Modules detected:", n_mods, "\n")

if (n_mods == 0) stop("No modules detected for ", DATASET)

B <- as.data.frame(t(MEs))
colnames(B) <- samples
rownames(B) <- gsub("ME", "M", rownames(B))

# kME loadings (Z: genes x modules)
kME <- signedKME(datExpr, MEs)
colnames(kME) <- gsub("kME", "M", colnames(kME))
rownames(kME) <- norm_genes

model_dir <- file.path(out_dir, "WGCNA")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(B,   file.path(model_dir, "B.csv"))
write.csv(kME, file.path(model_dir, "Z.csv"))
saveRDS(net, file.path(model_dir, "wgcna_network.rds"))
message("  WGCNA saved -> ", model_dir)